<a href="https://colab.research.google.com/github/cpohagwu/crosslearn/blob/main/examples/07_cityLearn_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CityLearn SAC with Chronos

This notebook shows two ways to add Chronos representations to a CityLearn + Stable-Baselines3 SAC workflow:

1. policy-side embeddings with `ChronosExtractor`
2. env-side walk-forward embeddings with `WalkForwardChronosWrapper`

The first path treats each normalized CityLearn observation as a one-step Chronos window. The second path lets the environment collect short observation histories before SAC sees them.


In [ ]:
# Uncomment in a fresh environment:
# IMPORTANT:
# First install CityLearn before CrossLearn to ensure compatibility, as CrossLearn's dependencies are more relaxed.
# Make sure to restart the kernel after installing or upgrading packages to ensure changes take effect.
# %pip install citylearn==2.6.0b2 --quiet
# %pip install -U crosslearn[chronos] --quiet

In [ ]:
import importlib.metadata

from stable_baselines3.sac import SAC as Agent

from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

from crosslearn.envs import WalkForwardChronosWrapper
from crosslearn.extractors import ChronosExtractor

for package_name in ['crosslearn', 'citylearn', 'stable-baselines3']:
    try:
        print(f'{package_name}: {importlib.metadata.version(package_name)}')
    except importlib.metadata.PackageNotFoundError:
        print(f'{package_name}: not installed')

## Wrapper References

CityLearn must use `central_agent=True` for SB3 because SB3 expects a single-agent interface. The CityLearn wrappers come first, then CrossLearn is added either through `policy_kwargs` or through an environment wrapper.

Reference docs: [CityLearn `NormalizedObservationWrapper`](https://www.citylearn.net/api/citylearn.wrappers.html#citylearn.wrappers.NormalizedObservationWrapper), [CityLearn `StableBaselines3Wrapper`](https://www.citylearn.net/api/citylearn.wrappers.html#citylearn.wrappers.StableBaselines3Wrapper), [SB3 SAC](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html), [CrossLearn Chronos guide](https://github.com/cpohagwu/crosslearn/blob/main/docs/chronos.md), [CrossLearn quickstarts](https://github.com/cpohagwu/crosslearn#quickstart-colab-notebooks).


In [ ]:
DATASET_NAME = 'citylearn_challenge_2023_phase_2_local_evaluation'
MODEL_NAME = 'amazon/chronos-bolt-tiny'
EPISODES = 2
SEED = 42

POLICY_LOOKBACK = 1
WRAPPER_LOOKBACK = 4
WRAPPER_MIN_HISTORY = 4

In [ ]:
def make_citylearn_sb3_env():
    env = CityLearnEnv(DATASET_NAME, central_agent=True)
    env = NormalizedObservationWrapper(env)
    env = StableBaselines3Wrapper(env)
    return env


def evaluate_citylearn_agent(env, model):
    observations, _ = env.reset()

    while not env.unwrapped.terminated:
        actions, _ = model.predict(observations, deterministic=True)
        observations, _, _, _, _ = env.step(actions)

    kpis = env.unwrapped.evaluate()
    kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
    return kpis.dropna(how='all')

## Policy-Side `ChronosExtractor`

This path keeps the CityLearn environment output unchanged after the SB3 wrapper. SAC receives those normalized vectors and `ChronosExtractor` embeds them inside the policy forward pass. With `lookback=1`, each CityLearn observation is treated as a one-step Chronos window.


In [ ]:
policy_env = make_citylearn_sb3_env()

policy_model = Agent(
    'MlpPolicy',
    policy_env,
    policy_kwargs={
        'features_extractor_class': ChronosExtractor,
        'features_extractor_kwargs': {
            'lookback': POLICY_LOOKBACK,
            'model_name': MODEL_NAME,
        },
    },
    verbose=1,
    seed=SEED,
)

policy_model.learn(total_timesteps=policy_env.unwrapped.time_steps * EPISODES)
policy_kpis = evaluate_citylearn_agent(policy_env, policy_model)
display(policy_kpis)

## Env-Side `WalkForwardChronosWrapper`

This path collects a short normalized-observation history inside the environment. SAC uses a plain `MlpPolicy` because the wrapped environment already returns Chronos vectors. The small `lookback=4` is for demonstration; you can increase it for more temporal context at the cost of more training time and memory usage.

See the reference docs above for more details on the wrappers and their parameters. The code below shows how to set up both paths in a CityLearn + SB3 SAC workflow.

In [ ]:
def make_walkforward_chronos_env():
    env = make_citylearn_sb3_env()
    env = WalkForwardChronosWrapper(
        env,
        lookback=WRAPPER_LOOKBACK,
        min_history=WRAPPER_MIN_HISTORY,
        model_name=MODEL_NAME,
    )
    return env


wrapper_env = make_walkforward_chronos_env()

wrapper_model = Agent(
    'MlpPolicy',
    wrapper_env,
    verbose=1,
    seed=SEED,
)

wrapper_model.learn(total_timesteps=wrapper_env.unwrapped.time_steps * EPISODES)
wrapper_kpis = evaluate_citylearn_agent(wrapper_env, wrapper_model)
display(wrapper_kpis)